**Objetivo:** Preparar as características de conectividade (`X`), os rótulos (`y`) e os grupos (`SITE_ID`) para a etapa de classificação por Machine Learning.

**Entradas:** `Data/conectivity/connectivity_matrices_abide_validado.pkl` (Caderno 05) e `Data/raw/phenotypic_abide.csv` (Caderno 02).

**Processamento:** Extração das conexões únicas das matrizes de conectividade 200 × 200, resultando em 19.900 características por participante. Os rótulos são recodificados para `0 = Controle` e `1 = TEA`. A validação cruzada é definida utilizando `StratifiedGroupKFold`, com 5 folds e `SITE_ID` como grupo.

O `StandardScaler` não é ajustado neste notebook. Qualquer transformação que aprenda parâmetros a partir dos dados deve ser ajustada exclusivamente no conjunto de treinamento de cada fold, no notebook de classificação, para evitar data leakage.

**Saída:** `Data/processed/dataset_abide_processado.pkl`.

In [1]:
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

Projeto (BASE_DIR) = /home/maria/TCC_TEA


In [2]:
# ============================================================
# CARREGAMENTO E PREPARAÇÃO DAS CARACTERÍSTICAS
# ============================================================

exigir_arquivo(
    PATHS['conectivity']
    / 'connectivity_matrices_abide_validado.pkl',
    '05_Validacao_ABIDE.ipynb'
)

exigir_arquivo(
    PATHS['raw']
    / 'phenotypic_abide.csv',
    '02_Download_ABIDE.ipynb'
)

import pickle
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Carregar matrizes de conectividade validadas
# ------------------------------------------------------------
conn_path = (
    PATHS['conectivity']
    / 'connectivity_matrices_abide_validado.pkl'
)

with open(conn_path, 'rb') as f:
    conn_dict = pickle.load(f)

conn_dict = {
    str(sub_id): np.asarray(mat)
    for sub_id, mat in conn_dict.items()
}

# ------------------------------------------------------------
# Carregar dados fenotípicos
# ------------------------------------------------------------
phenotypic = pd.read_csv(
    PATHS['raw'] / 'phenotypic_abide.csv'
)

phenotypic['SUB_ID'] = (
    phenotypic['SUB_ID']
    .astype(str)
)

phenotypic = phenotypic.set_index(
    'SUB_ID'
)

# ------------------------------------------------------------
# IDs dos participantes
# ------------------------------------------------------------
sub_ids = np.array(
    list(conn_dict.keys()),
    dtype=str
)

# ------------------------------------------------------------
# Conferir correspondência dos participantes
# ------------------------------------------------------------
ids_sem_fenotipo = [
    sub_id
    for sub_id in sub_ids
    if sub_id not in phenotypic.index
]

if ids_sem_fenotipo:
    raise ValueError(
        'Existem participantes com matriz de conectividade '
        'sem correspondente nos dados fenotípicos. '
        f'Quantidade: {len(ids_sem_fenotipo)}'
    )

# ------------------------------------------------------------
# Matrizes de conectividade
# ------------------------------------------------------------
matrices = np.stack(
    [conn_dict[sub_id] for sub_id in sub_ids]
)

# ------------------------------------------------------------
# Conferência da dimensão das matrizes
# ------------------------------------------------------------
if matrices.ndim != 3:
    raise ValueError(
        f'Formato inesperado das matrizes: {matrices.shape}'
    )

if matrices.shape[1:] != (200, 200):
    raise ValueError(
        'As matrizes de conectividade devem possuir '
        f'dimensão 200 × 200. Encontrado: {matrices.shape[1:]}'
    )

# ------------------------------------------------------------
# Extração das conexões únicas
#
# Matriz simétrica:
# 200 × 200
#
# Número de conexões sem diagonal:
# 200 × 199 / 2 = 19.900
# ------------------------------------------------------------
# Triângulo SUPERIOR sem a diagonal, em ordem de linha (mesma convenção do TDAH).
# Atenção: nilearn.sym_matrix_to_vec devolve o triângulo INFERIOR; a ordem dos 19.900 atributos
# é diferente. Como os notebooks 09/10 reconstroem a matriz a partir de X, a ordem precisa ser conhecida.
roi_i, roi_j = np.triu_indices(matrices.shape[1], k=1)
X = matrices[:, roi_i, roi_j].astype(np.float64)

# ------------------------------------------------------------
# Rótulos
#
# DX_GROUP:
# 1 = TEA
# 2 = Controle
# ------------------------------------------------------------
y_original = phenotypic.loc[
    sub_ids,
    'DX_GROUP'
].to_numpy()

# ------------------------------------------------------------
# Grupos para validação
#
# SITE_ID representa o site de aquisição.
# ------------------------------------------------------------
groups = phenotypic.loc[
    sub_ids,
    'SITE_ID'
].astype(str)
groups = phenotypic.loc[
    sub_ids,
    'SITE_ID'
].astype(str).to_numpy()

# ------------------------------------------------------------
# Conferências finais
# ------------------------------------------------------------
if X.shape[1] != 19900:
    raise ValueError(
        'A quantidade de características não é 19.900. '
        f'Encontrado: {X.shape[1]}'
    )

if len(X) != len(y_original) or len(X) != len(groups):
    raise ValueError(
        'X, y e groups possuem quantidades diferentes '
        'de participantes.'
    )

print('==========================================')
print('DADOS PREPARADOS')
print('==========================================')
print('Participantes:', len(sub_ids))
print('X:', X.shape)
print('y original:', y_original.shape)
print('Sites únicos:', len(np.unique(groups)))
print('Características por participante:', X.shape[1])
print('Esperado:', 19900)

FileNotFoundError: Arquivo phenotypic_abide.csv não encontrado.
Caminho esperado: /home/maria/TCC_TEA/Data/raw/phenotypic_abide.csv

Este arquivo é produzido pelo notebook:
02_Download_ABIDE.ipynb

Execute esse notebook antes de continuar.

In [ ]:
# ============================================================
# RECODIFICAÇÃO DOS RÓTULOS
# ============================================================
#
# DX_GROUP:
# 1 = TEA
# 2 = Controle
#
# Para os modelos:
# 0 = Controle
# 1 = TEA
# ============================================================

y_bin = (
    y_original == 1
).astype(int)

valores_validos = set(
    np.unique(y_original)
)

if not valores_validos.issubset({1, 2}):
    raise ValueError(
        'Foram encontrados valores inesperados em DX_GROUP: '
        f'{sorted(valores_validos)}'
    )

print('==========================================')
print('DISTRIBUIÇÃO DAS CLASSES')
print('==========================================')
print(
    'Controle (0):',
    int(np.sum(y_bin == 0))
)
print(
    'TEA (1):',
    int(np.sum(y_bin == 1))
)
print(
    'Total:',
    len(y_bin)
)

In [ ]:
# ============================================================
# VALIDAÇÃO CRUZADA ESTRATIFICADA POR GRUPO
# ============================================================
#
# StratifiedGroupKFold:
# - mantém a estratificação das classes quando possível;
# - impede que participantes do mesmo SITE_ID apareçam
#   simultaneamente em treino e teste;
# - utiliza 5 folds;
# - mantém SEED = 42 para reprodutibilidade.
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

folds = list(
    sgkf.split(
        X,
        y_bin,
        groups
    )
)

print('==========================================')
print('VALIDAÇÃO DOS FOLDS')
print('==========================================')

for i, (train_idx, test_idx) in enumerate(folds):

    sites_treino = set(
        groups[train_idx]
    )

    sites_teste = set(
        groups[test_idx]
    )

    intersecao = (
        sites_treino
        &
        sites_teste
    )

    classes_treino = np.unique(
        y_bin[train_idx]
    )

    classes_teste = np.unique(
        y_bin[test_idx]
    )

    print()
    print(
        f'Fold {i}: '
        f'treino={len(train_idx)} | '
        f'teste={len(test_idx)}'
    )

    print(
        f'  Classes treino: {classes_treino.tolist()}'
    )

    print(
        f'  Classes teste: {classes_teste.tolist()}'
    )

    print(
        f'  Sites em comum: {len(intersecao)}'
    )

    if len(intersecao) != 0:
        raise RuntimeError(
            f'ERRO: o Fold {i} possui sites '
            'presentes simultaneamente em treino e teste.'
        )

    if len(classes_teste) < 2:
        raise RuntimeError(
            f'ERRO: o Fold {i} não possui as duas classes '
            'no conjunto de teste.'
        )

print()
print(
    'OK: nenhum SITE_ID aparece simultaneamente '
    'em treino e teste.'
)

In [ ]:
# ============================================================
# SALVAR DATASET PROCESSADO
# ============================================================
#
# Este arquivo será utilizado pelo Caderno 08.
#
# IMPORTANTE:
# O StandardScaler não é salvo aqui.
# Qualquer transformação que aprenda parâmetros a partir dos
# dados deve ser ajustada somente no conjunto de treino de cada
# fold durante a classificação.
# ============================================================

dataset_processado = {
    'X': X,
    'y': y_bin,
    'groups': groups,
    'sub_ids': sub_ids,
    'folds': folds,
    'n_splits': N_SPLITS,
    'seed': SEED,
}

out_path = (
    PATHS['processed']
    / 'dataset_abide_processado.pkl'
)

with open(
    out_path,
    'wb'
) as f:
    pickle.dump(
        dataset_processado,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print('==========================================')
print('DATASET PROCESSADO SALVO')
print('==========================================')
print('Arquivo:', out_path)
print('X:', X.shape)
print('y:', y_bin.shape)
print('groups:', groups.shape)
print('Folds:', len(folds))
print('Seed:', SEED)

print()
print(
    'StandardScaler: não salvo. '
    'Deve ser ajustado dentro de cada fold no Caderno 08.'
)